In [3]:
balls = pd.read_csv("../data/raw/deliveries_cleaned.csv")
matches = pd.read_csv("../data/raw/matches_cleaned.csv")


LOAD DATASET

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/dataset.csv")
df.head()


,match_id,batter,runs,balls_faced,recent_form,career_avg
0,548376,a ashish reddy,4,5,7.0,12.173913
1,598000,a ashish reddy,7,4,6.4,12.173913
2,598004,a ashish reddy,14,12,8.6,12.173913
3,598010,a ashish reddy,16,9,10.2,12.173913
4,598013,a ashish reddy,4,5,9.0,12.173913


merging

In [4]:
matches.rename(columns={'id': 'match_id'}, inplace=True)

df = balls.merge(
    matches[['match_id', 'venue', 'date']],
    on='match_id',
    how='left'
)

df.head()


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder,venue,date
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,sc ganguly,p kumar,BB McCullum,0,1,1,legbyes,True,none,none,none,m chinnaswamy stadium,2008-04-18
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,bb mccullum,p kumar,SC Ganguly,0,0,0,none,True,none,none,none,m chinnaswamy stadium,2008-04-18
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,bb mccullum,p kumar,SC Ganguly,0,1,1,wides,True,none,none,none,m chinnaswamy stadium,2008-04-18
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,bb mccullum,p kumar,SC Ganguly,0,0,0,none,True,none,none,none,m chinnaswamy stadium,2008-04-18
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,bb mccullum,p kumar,SC Ganguly,0,0,0,none,True,none,none,none,m chinnaswamy stadium,2008-04-18


In [5]:
df.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs',
       'total_runs', 'extras_type', 'is_wicket', 'player_dismissed',
       'dismissal_kind', 'fielder', 'venue', 'date'],
      dtype='object')

AGGREGATE BALL-BY-BALL → PLAYER-MATCH LEVEL

In [6]:
player_match = (
    df.groupby(['match_id', 'batter', 'venue', 'batting_team', 'bowling_team', 'date'])['batsman_runs']
    .sum()
    .reset_index()
)

player_match.rename(columns={'batsman_runs': 'runs'}, inplace=True)
player_match.head()


,match_id,batter,venue,batting_team,bowling_team,date,runs
0,335982,aa noffke,m chinnaswamy stadium,Royal Challengers Bangalore,Kolkata Knight Riders,2008-04-18,9
1,335982,b akhil,m chinnaswamy stadium,Royal Challengers Bangalore,Kolkata Knight Riders,2008-04-18,0
2,335982,bb mccullum,m chinnaswamy stadium,Kolkata Knight Riders,Royal Challengers Bangalore,2008-04-18,158
3,335982,cl white,m chinnaswamy stadium,Royal Challengers Bangalore,Kolkata Knight Riders,2008-04-18,6
4,335982,dj hussey,m chinnaswamy stadium,Kolkata Knight Riders,Royal Challengers Bangalore,2008-04-18,12


SORT BY DATE (CRITICAL FOR TIME SERIES)

In [7]:
player_match['date'] = pd.to_datetime(player_match['date'])
player_match = player_match.sort_values(['batter', 'date'])


ENGINEER FEATURES (MAIN PART)
Rolling Average (Player Form – Last 5 Matches)

In [8]:
player_match['recent_form'] = (
    player_match.groupby('batter')['runs']
    .rolling(5)
    .mean()
    .reset_index(level=0, drop=True)
)


Venue Average

In [9]:
player_match['venue_avg'] = (
    player_match.groupby(['batter', 'venue'])['runs']
    .transform('mean')
)


Opponent-Specific Stats (PvT)

In [10]:
player_match['opponent_avg'] = (
    player_match.groupby(['batter', 'bowling_team'])['runs']
    .transform('mean')
)


Career Average

In [11]:
player_match['career_avg'] = (
    player_match.groupby('batter')['runs']
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)


STEP 5: CREATE TRAINING LABEL

In [12]:
player_match['target_runs'] = (
    player_match.groupby('batter')['runs']
    .shift(-1)
)


STEP 6: DROP NaNs (FIRST MATCHES & LAST MATCH)

In [13]:
model_df = player_match.dropna()
model_df.head()


,match_id,batter,venue,batting_team,bowling_team,date,runs,recent_form,venue_avg,opponent_avg,career_avg,target_runs
4747,548376,a ashish reddy,"rajiv gandhi international stadium, uppal",Deccan Chargers,Royal Challengers Bangalore,2012-05-20,4,7.0,8.454545,11.0,7.000000,7.0
4866,598000,a ashish reddy,"rajiv gandhi international stadium, uppal",Sunrisers Hyderabad,Pune Warriors,2013-04-05,7,6.4,8.454545,13.0,7.000000,14.0
4933,598004,a ashish reddy,"rajiv gandhi international stadium, uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,2013-04-07,14,8.6,8.454545,11.0,8.000000,3.0
5597,598048,a ashish reddy,m chinnaswamy stadium,Sunrisers Hyderabad,Royal Challengers Bangalore,2013-04-09,3,7.6,17.500000,11.0,7.375000,16.0
5027,598010,a ashish reddy,feroz shah kotla,Sunrisers Hyderabad,Delhi Daredevils,2013-04-12,16,8.8,16.000000,12.0,8.333333,4.0


STEP 7: TRAIN-TEST SPLIT (TIME-SERIES AWARE)

In [14]:
split_date = model_df['date'].quantile(0.8)

train_df = model_df[model_df['date'] <= split_date]
test_df = model_df[model_df['date'] > split_date]

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)


Train size: (11079, 12)
Test size: (2736, 12)


PREPROCESSING PIPELINE

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

num_features = [
    'recent_form',
    'venue_avg',
    'opponent_avg',
    'career_avg'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

pipeline.fit(train_df[num_features])

joblib.dump(pipeline, "../models/feature_pipeline.pkl")

print("Feature pipeline saved")


Feature pipeline saved


In [9]:
import os

BASE_DIR = os.path.abspath("..")
DATA_PROCESSED_PATH = os.path.join(BASE_DIR, "data", "processed")

print("Base dir:", BASE_DIR)
print("Processed path:", DATA_PROCESSED_PATH)

os.makedirs(DATA_PROCESSED_PATH, exist_ok=True)


Base dir: d:\cricket-performance-prediction
Processed path: d:\cricket-performance-prediction\data\processed
